In [1]:
import os
import json
from typing import Dict, List, Tuple, Optional

# ----------------------------
# 0) 基本配置
# ----------------------------
# 你的本地 datasets 缓存目录（你说数据下载在这里）
DATA_CACHE_DIR = "./local_sib200_data"
os.makedirs(DATA_CACHE_DIR, exist_ok=True)

# 如果你在国内环境需要镜像
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

# 让 datasets 优先使用你这个缓存目录（可选，但很建议）
os.environ.setdefault("HF_DATASETS_CACHE", os.path.abspath(DATA_CACHE_DIR))

REPO_ID = "mteb/sib200"

# 你要的 15 种语言（输出 JSON 用这些 key）
LANGS = ["ar", "bg", "de", "el", "en", "es", "fr", "hi", "ru", "sw", "th", "tr", "ur", "vi", "zh"]

# 语言 -> SIB200 config（按需修改；zh 也可改成 zho_Hant）
LANG_TO_CONFIG = {
    "ar": "arb_Arab",
    "bg": "bul_Cyrl",
    "de": "deu_Latn",
    "el": "ell_Grek",
    "en": "eng_Latn",
    "es": "spa_Latn",
    "fr": "fra_Latn",
    "hi": "hin_Deva",
    "ru": "rus_Cyrl",
    "sw": "swh_Latn",
    "th": "tha_Thai",
    "tr": "tur_Latn",
    "ur": "urd_Arab",
    "vi": "vie_Latn",
    "zh": "zho_Hans",
}

# split 名称候选：有的库叫 validation，有的叫 valid/dev
SPLIT_PLAN = {
    "train": ["train"],
    "validation": ["validation", "valid", "dev"],
    "test": ["test"],
}

# ----------------------------
# 1) 加载本地模型（你的 local_model）
# ----------------------------
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig

MODEL_PATH = "./local_model"

label2id = {
    "science/technology": 0,
    "travel": 1,
    "politics": 2,
    "sports": 3,
    "health": 4,
    "entertainment": 5,
    "geography": 6,
}
id2label = {v: k for k, v in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
config = AutoConfig.from_pretrained(
    MODEL_PATH,
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label,
    local_files_only=True,
)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    config=config,
    local_files_only=True,
)

device = (
    "mps" if torch.backends.mps.is_available()
    else ("cuda" if torch.cuda.is_available() else "cpu")
)
model = model.to(device).eval()
print(f"✅ Model loaded on {device}")


# ----------------------------
# 2) 批量预测：orig_pred=top1, targeted_pred=top2
# ----------------------------
@torch.no_grad()
def predict_top1_top2(texts: List[str], batch_size: int = 64) -> Tuple[List[int], List[int]]:
    orig_pred: List[int] = []
    targeted_pred: List[int] = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = tokenizer(batch, return_tensors="pt", truncation=True, padding=True)
        enc = {k: v.to(device) for k, v in enc.items()}

        logits = model(**enc).logits  # [B, num_labels]
        top2 = torch.topk(logits, k=2, dim=-1).indices.detach().cpu().tolist()  # [B, 2]

        orig_pred.extend([x[0] for x in top2])
        targeted_pred.extend([x[1] for x in top2])

    return orig_pred, targeted_pred


# ----------------------------
# 3) 数据加载：优先复用本地 cache_dir
# ----------------------------
from datasets import load_dataset

def try_load_split(repo_id: str, config_name: str, split_candidates: List[str]):
    last_err = None
    for sp in split_candidates:
        try:
            ds = load_dataset(repo_id, config_name, split=sp, cache_dir=DATA_CACHE_DIR)
            return sp, ds
        except Exception as e:
            last_err = e
    return None, last_err


# ----------------------------
# 4) pad 对齐导出
# ----------------------------
def export_one_split_pad(split_key: str, out_path: str):
    # 4.1 加载各语言 split
    datasets_by_lang: Dict[str, object] = {}
    used_split_name = None

    for lang in LANGS:
        cfg = LANG_TO_CONFIG[lang]
        sp, obj = try_load_split(REPO_ID, cfg, SPLIT_PLAN[split_key])
        if sp is None:
            raise RuntimeError(f"Split '{split_key}' not found for config={cfg}. Last error: {obj}")
        used_split_name = sp
        datasets_by_lang[lang] = obj

    lens = {lang: len(datasets_by_lang[lang]) for lang in LANGS}
    print(f"📏 Split={split_key} lengths: {lens}")

    # pad：取最大长度
    n = max(lens.values())

    # 4.2 各语言分别预测（只预测各语言自己有的行）
    preds_by_lang: Dict[str, Tuple[List[int], List[int]]] = {}
    for lang in LANGS:
        texts = datasets_by_lang[lang]["text"]
        orig, targ = predict_top1_top2(texts, batch_size=64)
        preds_by_lang[lang] = (orig, targ)
        print(f"✅ Pred done: {split_key}/{lang} ({len(texts)} rows)")

    # 4.3 安全取值（越界 -> None）
    def safe_get(ds, i: int, key: str):
        return ds[key][i] if i < len(ds) else None

    def safe_get_pred(lang: str, i: int, which: int) -> Optional[int]:
        arr = preds_by_lang[lang][which]
        return int(arr[i]) if i < len(arr) else None

    def get_label(i: int) -> Optional[int]:
        # 优先用英文的 label；如果英文越界，再找任意一个语言的 label
        if i < len(datasets_by_lang["en"]):
            return int(datasets_by_lang["en"]["label"][i])
        for lang in LANGS:
            v = safe_get(datasets_by_lang[lang], i, "label")
            if v is not None:
                return int(v)
        return None

    # 4.4 组装 JSON（缺失语言填 null）
    data = []
    for i in range(n):
        item = {
            "index": i,
            "text": {lang: safe_get(datasets_by_lang[lang], i, "text") for lang in LANGS},
            "label": get_label(i),
            "orig_pred": {lang: safe_get_pred(lang, i, 0) for lang in LANGS},
            "targeted_pred": {lang: safe_get_pred(lang, i, 1) for lang in LANGS},
        }
        data.append(item)

    # 4.5 写文件
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"🎉 Saved {len(data)} rows to {out_path} (split used='{used_split_name}', align_mode='pad')")


if __name__ == "__main__":
    export_one_split_pad("train", "./sib200_train.json")
    export_one_split_pad("validation", "./sib200_validation.json")
    export_one_split_pad("test", "./sib200_test.json")


/Users/yilongwang/anaconda3/envs/py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ./local_model and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model loaded on mps


Generating test split: 204 examples [00:00, 50084.17 examples/s]
Generating train split: 701 examples [00:00, 193269.38 examples/s]
Generating validation split: 99 examples [00:00, 27393.86 examples/s]
Generating validation split: 100%|██████████| 99/99 [00:00<00:00, 37697.33 examples/s]
'(ReadTimeoutError("HTTPSConnectionPool(host='hf-mirror.com', port=443): Read timed out. (read timeout=10)"), '(Request ID: b00a5a49-086b-477e-b9af-0407c38cedbc)')' thrown while requesting HEAD https://hf-mirror.com/datasets/mteb/sib200/resolve/03511ffeb3e1f7a6f4a744228953b4f317a527fe/sib200.py
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='hf-mirror.com', port=443): Read timed out. (read timeout=10)"), '(Request ID: e8285ab4-771d-4ad2-9069-3ce7561b7530)')' thrown while requesting HEAD https://hf-mirror.com/datasets/mteb/sib200/resolve/03511ffeb3e1f7a6f4a744228953b4f317a527fe/dataset_infos.json
Retrying in 1s [Retry 1/5].
Generating validation split: 100%|██████████| 99/99 [0

📏 Split=train lengths: {'ar': 701, 'bg': 701, 'de': 701, 'el': 701, 'en': 701, 'es': 701, 'fr': 701, 'hi': 701, 'ru': 701, 'sw': 701, 'th': 478, 'tr': 701, 'ur': 700, 'vi': 701, 'zh': 701}
✅ Pred done: train/ar (701 rows)
✅ Pred done: train/bg (701 rows)
✅ Pred done: train/de (701 rows)
✅ Pred done: train/el (701 rows)
✅ Pred done: train/en (701 rows)
✅ Pred done: train/es (701 rows)
✅ Pred done: train/fr (701 rows)
✅ Pred done: train/hi (701 rows)
✅ Pred done: train/ru (701 rows)
✅ Pred done: train/sw (701 rows)
✅ Pred done: train/th (478 rows)
✅ Pred done: train/tr (701 rows)
✅ Pred done: train/ur (700 rows)
✅ Pred done: train/vi (701 rows)
✅ Pred done: train/zh (701 rows)
🎉 Saved 701 rows to ./sib200_train.json (split used='train', align_mode='pad')
📏 Split=validation lengths: {'ar': 99, 'bg': 99, 'de': 99, 'el': 99, 'en': 99, 'es': 99, 'fr': 99, 'hi': 99, 'ru': 99, 'sw': 99, 'th': 58, 'tr': 99, 'ur': 99, 'vi': 99, 'zh': 99}
✅ Pred done: validation/ar (99 rows)
✅ Pred done: validati

'(ReadTimeoutError("HTTPSConnectionPool(host='hf-mirror.com', port=443): Read timed out. (read timeout=10)"), '(Request ID: 534132ba-1469-4dcb-ae91-d74e37d9b1c6)')' thrown while requesting HEAD https://hf-mirror.com/datasets/mteb/sib200/resolve/03511ffeb3e1f7a6f4a744228953b4f317a527fe/.huggingface.yaml
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='hf-mirror.com', port=443): Read timed out. (read timeout=10)"), '(Request ID: 638646e7-714d-406f-a26f-cb8f48c2ba3e)')' thrown while requesting HEAD https://hf-mirror.com/datasets/mteb/sib200/resolve/03511ffeb3e1f7a6f4a744228953b4f317a527fe/sib200.py
Retrying in 1s [Retry 1/5].


📏 Split=test lengths: {'ar': 204, 'bg': 99, 'de': 99, 'el': 99, 'en': 99, 'es': 99, 'fr': 99, 'hi': 99, 'ru': 99, 'sw': 99, 'th': 58, 'tr': 99, 'ur': 99, 'vi': 99, 'zh': 204}
✅ Pred done: test/ar (204 rows)
✅ Pred done: test/bg (99 rows)
✅ Pred done: test/de (99 rows)
✅ Pred done: test/el (99 rows)
✅ Pred done: test/en (99 rows)
✅ Pred done: test/es (99 rows)
✅ Pred done: test/fr (99 rows)
✅ Pred done: test/hi (99 rows)
✅ Pred done: test/ru (99 rows)
✅ Pred done: test/sw (99 rows)
✅ Pred done: test/th (58 rows)
✅ Pred done: test/tr (99 rows)
✅ Pred done: test/ur (99 rows)
✅ Pred done: test/vi (99 rows)
✅ Pred done: test/zh (204 rows)


ValueError: Column 'label' doesn't exist.

In [1]:
import os
import json
from typing import Dict, List, Tuple, Optional

# ----------------------------
# 0) 配置：本地 cache 目录
# ----------------------------
DATA_CACHE_DIR = "./local_sib200_data"
os.makedirs(DATA_CACHE_DIR, exist_ok=True)

# 如需镜像可保留
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
# 建议把 HF datasets cache 指到你这个目录（可选）
os.environ.setdefault("HF_DATASETS_CACHE", os.path.abspath(DATA_CACHE_DIR))

REPO_ID = "mteb/sib200"

LANGS = ["ar", "bg", "de", "el", "en", "es", "fr", "hi", "ru", "sw", "th", "tr", "ur", "vi", "zh"]
LANG_TO_CONFIG = {
    "ar": "arb_Arab",
    "bg": "bul_Cyrl",
    "de": "deu_Latn",
    "el": "ell_Grek",
    "en": "eng_Latn",
    "es": "spa_Latn",
    "fr": "fra_Latn",
    "hi": "hin_Deva",
    "ru": "rus_Cyrl",
    "sw": "swh_Latn",
    "th": "tha_Thai",
    "tr": "tur_Latn",
    "ur": "urd_Arab",
    "vi": "vie_Latn",
    "zh": "zho_Hans",
}

# ----------------------------
# 1) 加载本地模型
# ----------------------------
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig

MODEL_PATH = "./local_model"

label2id = {
    "science/technology": 0,
    "travel": 1,
    "politics": 2,
    "sports": 3,
    "health": 4,
    "entertainment": 5,
    "geography": 6,
}
id2label = {v: k for k, v in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
config = AutoConfig.from_pretrained(
    MODEL_PATH,
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label,
    local_files_only=True,
)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    config=config,
    local_files_only=True,
)

device = (
    "mps" if torch.backends.mps.is_available()
    else ("cuda" if torch.cuda.is_available() else "cpu")
)
model = model.to(device).eval()
print(f"✅ Model loaded on {device}")

# ----------------------------
# 2) 批量预测：top1 / top2
# ----------------------------
@torch.no_grad()
def predict_top1_top2(texts: List[str], batch_size: int = 64) -> Tuple[List[int], List[int]]:
    orig_pred: List[int] = []
    targeted_pred: List[int] = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = tokenizer(batch, return_tensors="pt", truncation=True, padding=True)
        enc = {k: v.to(device) for k, v in enc.items()}

        logits = model(**enc).logits
        top2 = torch.topk(logits, k=2, dim=-1).indices.detach().cpu().tolist()

        orig_pred.extend([x[0] for x in top2])
        targeted_pred.extend([x[1] for x in top2])

    return orig_pred, targeted_pred

# ----------------------------
# 3) 加载 test split（复用 cache_dir）
# ----------------------------
from datasets import load_dataset

def load_test_split(config_name: str):
    # split 参数用于选择数据集提供的 test/train/validation 等分割。[web:38]
    return load_dataset(REPO_ID, config_name, split="test", cache_dir=DATA_CACHE_DIR)

def get_text_column(ds) -> str:
    # 兼容性：大多数是 text；万一不是就兜底
    cols = getattr(ds, "column_names", [])
    if "text" in cols:
        return "text"
    if "sentence" in cols:
        return "sentence"
    raise ValueError(f"No text column found. columns={cols}")

def safe_get(ds, i: int, key: str):
    # 关键修复：先判断列是否存在，再判断 index 是否越界。[web:112]
    if key not in ds.column_names:
        return None
    return ds[key][i] if i < len(ds) else None

# ----------------------------
# 4) 导出 test（pad 对齐）
# ----------------------------
def export_test_pad(out_path: str = "./sib200_test.json"):
    # 4.1 加载所有语言 test
    datasets_by_lang = {}
    text_key_by_lang = {}
    lens = {}

    for lang in LANGS:
        cfg = LANG_TO_CONFIG[lang]
        ds = load_test_split(cfg)
        datasets_by_lang[lang] = ds
        text_key_by_lang[lang] = get_text_column(ds)
        lens[lang] = len(ds)

    print(f"Split=test lengths: {lens}")

    # pad：取最大长度
    n = max(lens.values())

    # 4.2 预测（每个语言只预测它实际有的样本）
    preds_by_lang: Dict[str, Tuple[List[int], List[int]]] = {}
    for lang in LANGS:
        ds = datasets_by_lang[lang]
        tkey = text_key_by_lang[lang]
        texts = ds[tkey]
        orig, targ = predict_top1_top2(texts, batch_size=64)
        preds_by_lang[lang] = (orig, targ)
        print(f"✅ Pred done: test/{lang} ({len(texts)} rows)")

    def safe_get_pred(lang: str, i: int, which: int) -> Optional[int]:
        arr = preds_by_lang[lang][which]
        return int(arr[i]) if i < len(arr) else None

    def get_label(i: int) -> Optional[int]:
        # 优先用 en；如果 en 没 label 或越界，就在其它语言里找“存在 label 列且不越界”的第一个
        en_ds = datasets_by_lang["en"]
        v = safe_get(en_ds, i, "label")
        if v is not None:
            return int(v)
        for lang in LANGS:
            v2 = safe_get(datasets_by_lang[lang], i, "label")
            if v2 is not None:
                return int(v2)
        return None

    # 4.3 组装 JSON（缺失填 null）
    data = []
    for i in range(n):
        item = {
            "index": i,
            "text": {
                lang: (
                    datasets_by_lang[lang][text_key_by_lang[lang]][i]
                    if i < len(datasets_by_lang[lang])
                    else None
                )
                for lang in LANGS
            },
            "label": get_label(i),
            "orig_pred": {lang: safe_get_pred(lang, i, 0) for lang in LANGS},
            "targeted_pred": {lang: safe_get_pred(lang, i, 1) for lang in LANGS},
        }
        data.append(item)

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"🎉 Saved {len(data)} rows to {out_path} (align_mode='pad')")

if __name__ == "__main__":
    export_test_pad("./sib200_test.json")


/Users/yilongwang/anaconda3/envs/py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ./local_model and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model loaded on mps
Split=test lengths: {'ar': 204, 'bg': 99, 'de': 99, 'el': 99, 'en': 99, 'es': 99, 'fr': 99, 'hi': 99, 'ru': 99, 'sw': 99, 'th': 58, 'tr': 99, 'ur': 99, 'vi': 99, 'zh': 204}
✅ Pred done: test/ar (204 rows)
✅ Pred done: test/bg (99 rows)
✅ Pred done: test/de (99 rows)
✅ Pred done: test/el (99 rows)
✅ Pred done: test/en (99 rows)
✅ Pred done: test/es (99 rows)
✅ Pred done: test/fr (99 rows)
✅ Pred done: test/hi (99 rows)
✅ Pred done: test/ru (99 rows)
✅ Pred done: test/sw (99 rows)
✅ Pred done: test/th (58 rows)
✅ Pred done: test/tr (99 rows)
✅ Pred done: test/ur (99 rows)
✅ Pred done: test/vi (99 rows)
✅ Pred done: test/zh (204 rows)
🎉 Saved 204 rows to ./sib200_test.json (align_mode='pad')
